# Building One Row Per Customer with SQL

**The business question.** Customer data lives in separate tables: one row per customer in one, one row per order in another. Before anyone can segment customers, someone has to build a table with **one row per customer** that summarises how each one behaves. That job is done with two SQL ideas, `GROUP BY` and `JOIN`, and one wrong choice of join can quietly remove customers from your analysis without a single error message.

**About the data.** The Marlow & Finch tables are **synthetic**: a fictional home-goods retailer with 1,200 customers and about 6,750 orders, generated for teaching. Nobody in them is real.

**How this notebook works.** There are two parts. Each follows the same five steps:

1. Load and look
2. Run the query
3. Read the output in plain English
4. Check against pandas
5. Say it to a manager

SQL runs here inside a small in-memory database, so there is nothing to install or connect to. Copilot is switched on: in the second part you will ask it to write queries, and you will check what it writes against pandas. There is a Stretch space at the end.

## Set up

In [ ]:
import sqlite3

import numpy as np
import pandas as pd

pd.set_option("display.width", 140)

REFERENCE_DATE = "2025-07-01"      # recency is counted in days up to this date

customers = pd.read_csv("../data/customers.csv")
orders = pd.read_csv("../data/orders.csv")

con = sqlite3.connect(":memory:")                  # a small database that lives in memory
customers.to_sql("customers", con, index=False)    # one table per file
orders.to_sql("orders", con, index=False)

def sql(query, **params):
    """Run a query and return the result as a table."""
    return pd.read_sql(query, con, params=params)

print("Database ready with tables: customers and orders")

---
# Part 1. Demo: GROUP BY and JOIN

## Step 1. Load and look

Start by asking the database what it holds. `SELECT` chooses columns, `FROM` says which table, `COUNT(*)` counts rows.

In [ ]:
print(sql("SELECT COUNT(*) AS Customers FROM customers").to_string(index=False))
print(sql("SELECT COUNT(*) AS Orders FROM orders").to_string(index=False))
print()
print(sql("SELECT * FROM customers LIMIT 3").to_string(index=False))
print()
print(sql("SELECT * FROM orders LIMIT 3").to_string(index=False))

**What to look for.** The two tables share one column, `CustomerID`. That shared column is how rows in one table are matched to rows in the other.

## Step 2. Run the query

### GROUP BY: collapse many rows into one per group

`GROUP BY` splits the rows into groups and gives you one result row per group. You choose what to calculate for each group with an **aggregate function**: `COUNT`, `SUM`, `AVG`, `MIN`, `MAX`.

In [ ]:
by_region = sql("""
SELECT Region,
       COUNT(*) AS Customers
FROM customers
GROUP BY Region
ORDER BY Customers DESC
""")
print(by_region.to_string(index=False))

In [ ]:
by_category = sql("""
SELECT Category,
       COUNT(*)                    AS Orders,
       ROUND(AVG(OrderValue), 2)   AS AvgOrderValue,
       ROUND(SUM(OrderValue), 2)   AS Revenue
FROM orders
GROUP BY Category
ORDER BY Revenue DESC
""")
print(by_category.to_string(index=False))

### JOIN: bring two tables together

A `JOIN` matches rows from two tables on a shared column. There are two kinds you need to tell apart:

- `INNER JOIN` keeps only customers that have a match in the orders table.
- `LEFT JOIN` keeps **every** customer from the left-hand table and fills in empty values (called NULL) where there is no match.

Here is the table we need for segmentation: one row per customer, with recency, number of orders, average order value, discount share and total spend. The query is the same twice, and only the join word changes.

In [ ]:
one_row_query = """
SELECT c.CustomerID,
       c.Region,
       CAST(julianday(:ref) - julianday(MAX(o.OrderDate)) AS INTEGER) AS RecencyDays,
       COUNT(o.OrderID)   AS Orders,
       AVG(o.OrderValue)  AS AvgOrder,
       AVG(o.Discounted)  AS DiscountShare,
       SUM(o.OrderValue)  AS TotalSpend
FROM customers AS c
{join} JOIN orders AS o ON c.CustomerID = o.CustomerID
GROUP BY c.CustomerID, c.Region
"""

with_left = sql(one_row_query.format(join="LEFT"), ref=REFERENCE_DATE)
with_inner = sql(one_row_query.format(join="INNER"), ref=REFERENCE_DATE)

print("Rows with LEFT JOIN: ", len(with_left))
print("Rows with INNER JOIN:", len(with_inner))
print("Customers silently dropped by the inner join:", len(with_left) - len(with_inner))

## Step 3. Read the output in plain English

The inner join quietly returned fewer customers, and it did so without an error or a warning. Who are the missing ones? A query that finds rows with no match uses `LEFT JOIN` and then keeps the rows where the right-hand side is empty (`IS NULL`).

In [ ]:
never_bought = sql("""
SELECT c.CustomerID, c.Region, c.AcquisitionChannel, c.LoyaltyMember
FROM customers AS c
LEFT JOIN orders AS o ON c.CustomerID = o.CustomerID
WHERE o.OrderID IS NULL
""")
print("Customers with no orders:", len(never_bought))
print()
print("By acquisition channel:")
print(never_bought["AcquisitionChannel"].value_counts().to_string())
print()
print("In the LEFT JOIN table, what do these customers look like?")
print(with_left[with_left["Orders"] == 0].head(3).to_string(index=False))

**What to look for.**

- With `LEFT JOIN` the customers who never ordered stay in, with 0 orders and empty (NULL) values for everything that depends on an order, such as recency. You then have to decide what to do with them.
- With `INNER JOIN` they are gone. Anything you calculate afterwards, such as the share of customers in a segment, uses the wrong total, and nothing tells you.

## Step 4. Check against pandas

pandas can do the same job with `groupby` and `merge`. If the two tools disagree, one of them has a mistake.

In [ ]:
order_totals = orders.groupby("CustomerID").agg(Orders=("OrderID", "count"), TotalSpend=("OrderValue", "sum"))
merged_left = customers.merge(order_totals, on="CustomerID", how="left")
merged_inner = customers.merge(order_totals, on="CustomerID", how="inner")

rows = [("Rows, LEFT JOIN", len(with_left), len(merged_left)),
        ("Rows, INNER JOIN", len(with_inner), len(merged_inner)),
        ("Total spend", round(with_left["TotalSpend"].sum(), 2), round(merged_left["TotalSpend"].sum(), 2))]

def show(value):
    return f"{value:,.2f}" if isinstance(value, float) else f"{value:,}"

compare = pd.DataFrame({
    "Check": [r[0] for r in rows],
    "SQL": [show(r[1]) for r in rows],
    "pandas": [show(r[2]) for r in rows],
    "Match": [r[1] == r[2] for r in rows],
})
print(compare.to_string(index=False))

## Step 5. Say it to a manager

Two sentences to the head of marketing. Say why the choice between `INNER JOIN` and `LEFT JOIN` matters when you build a customer table for segmentation, and what you would do with the customers who have never ordered.

**Your answer:**

---
# Part 2. Exercise: three business questions

## Step 1. Load and look

Three managers have three questions. You will ask Copilot to write the query for each one. **Before you run anything, read the query aloud** (or to a neighbour) and say in plain English what you think it does. Then run it, and check the result against pandas.

1. *"How many orders, and how much revenue, does each region produce?"* (the region lives in `customers`, the money in `orders`). Name the three columns `Region`, `Orders` and `Revenue`.
2. *"Which customers have placed more than 12 orders?"* (hint for Copilot: a condition on a group needs `HAVING`, not `WHERE`). Return `CustomerID` and the number of orders, named `Orders`.
3. *"What share of orders in each acquisition channel used a discount?"* Name the two columns `AcquisitionChannel` and `DiscountShare`, with the share as a number between 0 and 1.

The tables you can use are `customers` (CustomerID, JoinDate, Region, AcquisitionChannel, LoyaltyMember) and `orders` (OrderID, CustomerID, OrderDate, Category, OrderValue, Discounted, Returned). `Discounted` is 1 when a discount was used and 0 when not.

In [ ]:
print(sql("PRAGMA table_info(customers)")[["name", "type"]].to_string(index=False))
print()
print(sql("PRAGMA table_info(orders)")[["name", "type"]].to_string(index=False))

## Step 2. Run the query

**Ask Copilot.** A prompt such as this works well:

> I have two SQLite tables, customers(CustomerID, JoinDate, Region, AcquisitionChannel, LoyaltyMember) and orders(OrderID, CustomerID, OrderDate, Category, OrderValue, Discounted, Returned). Write a SQL query that returns the number of orders and the total OrderValue for each Region. Name the columns Region, Orders and Revenue.

Paste each query between the triple quotes below and run the cell.

In [ ]:
q1 = """

"""
if q1.strip():
    result1 = sql(q1)
    print(result1.to_string(index=False))
else:
    print("Paste the query Copilot wrote between the quotes above, then run this cell again.")

In [ ]:
q2 = """

"""
if q2.strip():
    result2 = sql(q2)
    print("Rows returned:", len(result2))
    print(result2.head().to_string(index=False))
else:
    print("Paste the query Copilot wrote between the quotes above, then run this cell again.")

In [ ]:
q3 = """

"""
if q3.strip():
    result3 = sql(q3)
    print(result3.to_string(index=False))
else:
    print("Paste the query Copilot wrote between the quotes above, then run this cell again.")

## Step 3. Read the output in plain English

For each question, write one sentence that answers the manager in business language, using the numbers in the table. Then ask yourself which join Copilot chose and whether it was the right one for the question. (For these three questions, think about whether a customer with no orders should appear.)

**Your answer:**

## Step 4. Check against pandas

The cell below works out the same three answers with pandas. The checks only run for queries you have pasted in.

In [ ]:
joined = orders.merge(customers, on="CustomerID", how="inner")

expected1 = joined.groupby("Region").agg(Orders=("OrderID", "count"), Revenue=("OrderValue", "sum")).round(2)
per_customer = orders.groupby("CustomerID").size()
expected2 = per_customer[per_customer > 12]
expected3 = joined.groupby("AcquisitionChannel")["Discounted"].mean().round(3)

def report(label, ok):
    print(f"{label}: {'matches pandas' if ok else 'does NOT match pandas, look again'}")

if q1.strip():
    got = result1.set_index("Region")
    report("Question 1, orders and revenue by region",
           got["Orders"].sort_index().tolist() == expected1["Orders"].sort_index().tolist()
           and np.allclose(got["Revenue"].sort_index(), expected1["Revenue"].sort_index()))
if q2.strip():
    report("Question 2, customers with more than 12 orders", len(result2) == len(expected2))
if q3.strip():
    got = result3.set_index("AcquisitionChannel")["DiscountShare"].sort_index()
    report("Question 3, discount share by channel", np.allclose(got, expected3.sort_index()))
if not (q1.strip() and q2.strip() and q3.strip()):
    print("Some queries are still empty above, so some checks were skipped.")

If a query does not match, do not just ask Copilot to fix it. Compare the row counts first: a wrong join or a missing `GROUP BY` usually shows up there.

## Step 5. Say it to a manager

Two sentences. Pick one of the three questions and give the manager the answer and one thing it suggests they should do.

**Your answer:**

---
# Stretch

Quick finishers: use Copilot to write each query and check it with pandas.

- Add a `CategoryBreadth` column to the one-row-per-customer query: the number of different product categories each customer has bought from (`COUNT(DISTINCT o.Category)`). Who buys from the most categories?
- Find the customers whose last order was more than 270 days before the reference date but who have placed more than five orders in total. They used to be good customers.

In [ ]:
# Your space.